# Theoretical Basis

`PyCBA` is based on the matrix stiffness method for linear elastic analysis, extended with an incremental nonlinear capability using the Generalized Clough model for concentrated plasticity. This page summarises the underlying theory.

For detailed treatments of the matrix stiffness method, see:
- Weaver, W. & Gere, J.M. (1990). *Matrix Analysis of Framed Structures*, 3rd ed. Springer.
- Ghali, A., Neville, A.M. & Brown, T.G. (2017). *Structural Analysis: A Unified Classical and Matrix Approach*, 7th ed. CRC Press.
- Timoshenko, S.P. & Young, D.H. (1965). *Theory of Structures*, 2nd ed. McGraw-Hill.

## Elements
The stiffness matrices for the elements used in `PyCBA` are derived from the Euler–Bernoulli beam theory with cubic Hermite shape functions (see Weaver & Gere, Chapter 5; Ghali & Neville, Chapter 5). The four element types correspond to the four combinations of end releases.

Note that the releases relate only to internal hinges within the elements themselves, and not to the supports conditions. Note also, that in modelling joints, there must always be one element with a 'fixed' end at a joint; otherwise the resulting global stiffness matrix will be singular.

### Type 1: Fixed-Fixed
$$
\bf{K} = \frac{EI}{L^3} 
\begin{bmatrix}
12 & 6L & -12 & 6L \\
6L & 4L^2 & -6L & 2L^2 \\
-12 & -6L & 12 & -6L \\
6L & 2L^2 & -6L & 4L^2
\end{bmatrix}
$$

### Type 2: Fixed-Pinned
$$
\bf{K} = \frac{3EI}{L^3} 
\begin{bmatrix}
1 & L & -1 & 0 \\
L & L^2 & -L & 0 \\
-1 & -L & 1 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

### Type 3: Pinned-Fixed
$$
\bf{K} = \frac{3EI}{L^3} 
\begin{bmatrix}
1 & 0 & -1 & L \\
0 & 0 & 0 & 0 \\
-1 & 0 & 1 & -L \\
L & 0 & -L & L^2
\end{bmatrix}
$$

### Type 4: Pinned-Pinned
$$
\bf{K} = 
\begin{bmatrix}
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

## Displacements

Displacement along an element are found using cumulative trapezoidal integration to approximate the integral. It is not accurate to use the beam element shape functions due to the presence of inter-nodal loading. Thus, the rotations along an element are found by integrating the curvatures:

$$
\theta(x) = \theta_0 + \int_{x=0}^{x=L} \frac{M(x)}{EI} dx,
$$

and similarly the displacements then found by integrating the rotations:

$$
\delta(x) = \delta_0 + \int_{x=0}^{x=L} \theta(x) dx.
$$

In both cases we approximate the integral with:

$$
\int_0^L y(x) dx \approx \sum_{i=1}^{i=n} \frac{y_{i-1}+y_i}{2} \Delta x.
$$

which is implemented using `SciPy`'s [cumulative trapezoidal](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.cumulative_trapezoid.html#scipy.integrate.cumulative_trapezoid) integration along the $n$ integration points along the member.

Note that $\theta_0$ and $\delta_0$ are required: the rotation and translation at the $i$-node of the element. This is typically the output of the global system of equations; i.e., the nodal displacements. However, for elements with releases (types 2, 3, 4 above), we need to determine the starting rotation, $\theta_0$. From the slope deflection equations we can find this starting rotation at the $i$-node:

$$
\theta_i = \frac{\delta_j - \delta_i}{L} - \frac{L}{3EI}\left(-FEM_i + 0.5FEM_j + M_i - 0.5 M_j\right)
$$

in which the $FEM$ are the consistent nodal loading moments due to the inter-nodal loads, and the $M$ moments are solved from the global stiffness equation.

## Global Stiffness Matrix

For an $N$-span beam there are $N+1$ nodes. Each node has two degrees of freedom (DOFs): a vertical displacement $v$ and a rotation $\theta$. The global DOF vector is of length $2(N+1)$, ordered node by node:

$$\mathbf{d} = \begin{bmatrix} v_0 & \theta_0 & v_1 & \theta_1 & \cdots & v_N & \theta_N \end{bmatrix}^T$$

The unrestricted global stiffness matrix $\mathbf{K}$ of size $2(N+1) \times 2(N+1)$ is assembled by the direct stiffness method. For span $m$ (zero-indexed), the left node is global node $m$ and the right node is global node $m+1$, mapping to global DOF indices $[2m,\; 2m+1,\; 2m+2,\; 2m+3]$. The $4\times 4$ element stiffness matrix $\mathbf{k}^{(e)}$ (one of the four types above) is overlapped into $\mathbf{K}$:

$$K_{[2m:2m+4],\,[2m:2m+4]} \mathrel{+}= \mathbf{k}^{(e)}$$

The overlap at interior nodes automatically enforces displacement compatibility between adjacent spans. The result is the banded, symmetric, positive semi-definite matrix characteristic of beam structures.

### Spring Supports

A spring support at DOF $i$ with stiffness $k_s > 0$ contributes a restoring force $k_s u_i$ at that DOF. This is incorporated directly into the global stiffness matrix before applying boundary conditions:

$$K_{ii} \leftarrow K_{ii} + k_s$$

A spring DOF remains as a free unknown in the linear system — no row or column elimination is performed. The spring force is recovered after solving as:

$$F_s^{(i)} = k_s \, u_i$$

## Boundary Conditions

The governing system of equations is:

$$\mathbf{K}\,\mathbf{d} = \mathbf{f}$$

where $\mathbf{f}$ is the equivalent nodal force vector assembled from the consistent nodal loads of each span (negated, as loads oppose the reactions they generate).

Boundary conditions are applied by the **direct elimination method**. A DOF $i$ is eliminated whenever its displacement is known: either because an explicit value $\bar{d}_i$ has been prescribed in the displacement vector $\mathbf{D}$, or because the DOF is fully fixed ($R_i = -1$) and no override is given, in which case $\bar{d}_i = 0$. For each such DOF the procedure is:

1. Transfer the constraint's contribution to all other DOFs:
$$f_j \leftarrow f_j - K_{ji}\,\bar{d}_i \quad \forall\, j$$

2. Zero the $i$-th row and column:
$$K_{ij} = K_{ji} = 0 \quad \forall\, j$$

3. Set the diagonal to unity and the right-hand side to the prescribed value:
$$K_{ii} = 1, \qquad f_i = \bar{d}_i$$

This preserves symmetry and reproduces the prescribed displacement exactly in the solution. A support settlement is therefore modelled simply by providing a non-zero $\bar{d}_i$ at a fixed DOF; a negative value corresponds to a downward settlement. Prescribed displacements may also be applied to free DOFs. Spring DOFs ($R_i > 0$) without a prescribed displacement are *not* eliminated; their stiffness is already on the diagonal and they remain as free unknowns.

## Reaction Recovery

After solving the restricted system for the nodal displacements $\mathbf{d}$, support reactions are recovered using the **unrestricted** stiffness matrix $\mathbf{K}_U$ (assembled before boundary conditions were applied) and the original nodal force vector $\mathbf{f}_U$:

$$\mathbf{r}^* = \mathbf{K}_U\,\mathbf{d} - \mathbf{f}_U$$

The reaction at a fully-fixed DOF $i$ ($R_i = -1$) is:

$$R_i = r^*_i$$

For spring DOFs the spring force is reported separately as $F_s^{(i)} = -k_s\,u_i$ (upward positive), because the residual $r^*_i$ at a spring DOF contains structural coupling contributions from neighbouring DOFs in addition to the spring force itself.

## Nonlinear Analysis — Generalized Clough Model

PyCBA extends the linear elastic framework with an incremental nonlinear analysis that tracks plastic hinge formation and moment redistribution up to collapse. The approach is based on *concentrated plasticity*: nonlinear behaviour is localised at element ends (potential hinge locations), while the element interior remains elastic.

### Concentrated Plasticity via Stiffness Degradation

The Generalized Clough model (Clough & Johnston, 1966) introduces a stiffness-reduction parameter $R$ at each element end. The parameter varies between:

- $R = 1$: fully elastic (no yielding)
- $R = q$: fully plastic hinge ($q$ is the strain-hardening ratio; $q = 0$ for elastic-perfectly-plastic)

The transition is governed by the normalised moment ratio $\gamma = |M| / M_p$:

$$
R = \begin{cases}
1 & \text{if } \gamma \leq \gamma_y \\[4pt]
1 - \dfrac{\gamma - \gamma_y}{1 - \gamma_y}(1 - q) & \text{if } \gamma_y < \gamma < 1 \\[4pt]
q & \text{if } \gamma \geq 1
\end{cases}
$$

where $\gamma_y = M_y / M_p$ is the normalised yield moment. This produces a bilinear moment–rotation response at the section level. On unloading ($\gamma$ decreasing below the historical peak), $R$ resets to 1 — the Clough "origin-oriented" unloading rule.

### Element Stiffness with Degradation

For an element with left-end parameter $R_1$ and right-end parameter $R_2$, the element stiffness matrix is interpolated between the standard fixed-fixed ($\mathbf{k}_e$), pinned-fixed ($\mathbf{k}_1$), and fixed-pinned ($\mathbf{k}_2$) matrices. Following Li & Li (2007, Chapter 4), if $R_1 \geq R_2$:

$$\mathbf{k} = R_2\,\mathbf{k}_e + (R_1 - R_2)\,\mathbf{k}_2$$

and if $R_1 < R_2$:

$$\mathbf{k} = R_1\,\mathbf{k}_e + (R_2 - R_1)\,\mathbf{k}_1$$

This interpolation ensures that the element stiffness degrades smoothly as either end yields, and reduces to the standard elastic matrix when $R_1 = R_2 = 1$.

### Hinge Ownership

At a shared node between two elements, each plastic hinge is *owned* by a single element end — the element whose end reached plasticity first. The adjacent element retains $R = 1$ at that shared node. This is essential: if both element ends at a node were simultaneously degraded, the global stiffness matrix would become singular prematurely, terminating the analysis before the true collapse mechanism forms.

The owning element is chosen as follows: for an interior node $j$, the hinge is assigned to element $j-1$ (right end). This convention keeps the global stiffness matrix non-singular during the progressive formation of hinges.

### Incremental Analysis (Static)

The proportional-load analysis proceeds incrementally:

1. **Initialise:** Set all $R = 1$, $\lambda = 0$ (load factor), $\mathbf{M} = \mathbf{0}$ (moments).

2. **Increment:** Choose a step $\Delta\lambda$ (adaptive: smaller steps when any $R$ is low, i.e. near plasticity).

3. **Solve:** Assemble $\mathbf{K}$ from the current $R$ values, apply boundary conditions, and solve:
$$\mathbf{K}\,\Delta\mathbf{u} = \Delta\lambda\,\mathbf{f}_{\text{ref}}$$
where $\mathbf{f}_{\text{ref}}$ is the reference load vector.

4. **Update moments:** $\mathbf{M} \leftarrow \mathbf{M} + \Delta\mathbf{M}(\Delta\mathbf{u})$

5. **Update R:** Recompute $\gamma$ at each node. If $\gamma$ exceeds the historical peak, degrade $R$ accordingly. If $\gamma$ has decreased (unloading), reset $R = 1$.

6. **Check for collapse:** If a new plastic hinge has formed, test whether the current hinge pattern constitutes a mechanism (see below).

7. **Repeat** until collapse or $\lambda_{\max}$ is reached.

### Moving Load Analysis

For a vehicle traversing the beam, the load position changes at each step. The key challenge is that the load must be *transferred* from the old position to the new one without introducing spurious plastic deformation. PyCBA uses paired **elastic-unload / nonlinear-reload** sub-increments at each position step:

1. **Unload** the previous position's load fraction using the *elastic* stiffness matrix (unloading is always elastic in the Clough model).
2. **Reload** the new position's load fraction using the *current* (degraded) stiffness matrix.

Each position step is divided into $n_{\text{sub}}$ sub-increments for accuracy. This paired unload/reload procedure ensures that elastic unloading does not trigger spurious yielding, while correctly accumulating plastic damage as the vehicle traverses.

### Collapse Detection

A collapse mechanism requires sufficient plastic hinges to render the structure kinematically unstable. PyCBA detects this by a **rank test**:

1. **Cluster** nearby hinged nodes (within a tolerance of $2 h_{\min}$, where $h_{\min}$ is the smallest element length) to account for mesh discretisation.

2. At each clustered hinge location, set $R = 0$ at *both* element ends meeting at that node — this fully releases the node, as would occur in a true mechanism.

3. Assemble the test stiffness matrix $\mathbf{K}_{\text{test}}$ and apply boundary conditions.

4. If $\operatorname{rank}(\mathbf{K}_{\text{test}}) < n_{\text{dof}}$, the structure is a mechanism and collapse has occurred.

This approach is more robust than monitoring determinant magnitude or displacement growth, as it directly tests for kinematic instability.

### Mesh Considerations

The continuous beam is internally meshed into short elements of a target length (`mesh_size`). The mesh density affects accuracy because:

- **Point loads** coinciding with mesh nodes are represented exactly. For loads between nodes, Hermite shape-function interpolation distributes the load to adjacent nodes.
- **UDL** is converted to lumped nodal forces (half the tributary load at each node of each element).
- **Plastic hinges** can only form at mesh nodes. For UDL, the theoretical hinge location (e.g. at $x^* = 0.414\,L$ for a propped cantilever) may not coincide with a node, causing the hinge to "snap" to a nearby node. This introduces a discretisation error that is *not* monotonically decreasing with mesh refinement — it depends on how close a node falls to the theoretical location.

For point-load problems where the load and hinge locations coincide with mesh nodes, the method converges rapidly. For UDL problems, errors of a few percent are typical and can be reduced by choosing mesh sizes that place nodes near the expected hinge locations.

## References

### Matrix Stiffness Method

- Weaver, W. & Gere, J.M. (1990). *Matrix Analysis of Framed Structures*, 3rd ed. Springer.
- Ghali, A., Neville, A.M. & Brown, T.G. (2017). *Structural Analysis: A Unified Classical and Matrix Approach*, 7th ed. CRC Press.
- Timoshenko, S.P. & Young, D.H. (1965). *Theory of Structures*, 2nd ed. McGraw-Hill.
- McGuire, W., Gallagher, R.H. & Ziemian, R.D. (2000). *Matrix Structural Analysis*, 2nd ed. John Wiley & Sons.

### Nonlinear Analysis and Plastic Hinge Models

- Clough, R.W. & Johnston, S.B. (1966). "Effect of stiffness degradation on earthquake ductility requirements." *Proc. Japan Earthquake Engineering Symposium*, Tokyo, pp. 227–232.
- Li, G.Q. & Li, J.J. (2007). *Advanced Analysis and Design of Steel Frames*. John Wiley & Sons, Chapter 4.
- Neal, B.G. (1977). *The Plastic Methods of Structural Analysis*, 3rd ed. Chapman & Hall.

### Plastic Analysis (Virtual Work / Collapse Load Factors)

- Baker, J., Horne, M.R. & Heyman, J. (1956). *The Steel Skeleton*, Vol. 2: Plastic Behaviour and Design. Cambridge University Press.
- Heyman, J. (1971). *Plastic Design of Frames*, Vol. 1: Fundamentals. Cambridge University Press.
- Horne, M.R. (1979). *Plastic Design of Low-Rise Frames*. Granada.

### Bridge Loading and Moving Load Analysis

- McCarthy, L.A. (2012). *Probabilistic Analysis of Indeterminate Highway Bridges Considering Material Nonlinearity*. MPhil Thesis, Dublin Institute of Technology.
- Caprani, C.C. (2006). "Probabilistic Analysis of Highway Bridge Traffic Loading." PhD Thesis, University College Dublin.